# Lecture 5: EDA and Data Quality
### Module 2 | LO2: Clean, summarize, visualize and interpret business datasets

## Central question
> **How does data quality affect the reliability of business insights?**

## The Case: Reliance Industries (Reliance Retail) 🏬
**Reliance Industries Limited** is one of India's largest companies, with businesses in energy, telecom (Jio) and retail.
**Reliance Retail** runs thousands of stores such as *Reliance Fresh, JioMart, Reliance Digital and Trends*, and every store sends its transactions to a central sales dashboard.

In the July to September 2026 quarter, the regional head noticed that **the dashboard's sales total did not match the finance department's number**, that **one city appeared under several different names**, and that **customer ratings, ages and incomes had many blanks**.
You are the junior analyst. You receive **200 transactions from four metro cities**. Before the head acts on any number, you must **check how clean the data is, fix it with Pandas, and show how much the insights change.**

> Data note: `Reliance_Retail_Transactions_v2.xlsx` is **made-up data** for teaching, not real Reliance data. It contains deliberate data-quality problems.

## How the three files map to each other
| File | What it holds |
|---|---|
| `Reliance_Data_Quality_Case.docx` | The case, business questions BQ1 to BQ7, dataset description, exercise and visualisation questions |
| `Reliance_Retail_Transactions_v2.xlsx` | Sheets: **Transactions** (200 rows, 14 columns), **Customer_Sample** (12 customers for the missing-value demos), **Data_Dictionary**, **Case_Map** |
| `Lecture5_Data_Quality_Reliance.ipynb` (this file) | Sections 1 to 7, with the business question written above every code cell |

## The 7 Business Questions (the same table is in the Word file and in the sheet `Case_Map`)
| ID | Business question | Topic | Notebook section | Data used (sheet: columns) |
|---|---|---|---|---|
| **BQ1** | Why are values missing, and which handling method suits each column? | Missing values: MCAR, MAR, MNAR and 5 handling methods | Section 1 | Customer_Sample (all columns); Transactions: Customer_Age, Monthly_Income, Customer_Rating, Store_City, Customer_Type, Payment_Mode |
| **BQ2** | Are any transactions counted twice, and how much is sales overstated? | Duplicate records | Section 2 | Transactions: Txn_ID, Sales_Amount (all columns) |
| **BQ3** | Are cities, categories and payment modes written in one standard way? | Inconsistent entries | Section 3A | Transactions: Store_City, Category, Payment_Mode, Customer_Type |
| **BQ4** | Are there impossible values (negative quantity, rating outside 1 to 5, zero price, wrong dates, amounts that do not add up)? | Incorrect entries | Section 3B | Transactions: Unit_Price, Quantity, Discount_Pct, Customer_Rating, Txn_Date, Sales_Amount |
| **BQ5** | Can we fix everything step by step with Pandas and keep a record of the changes? | Data cleaning using Pandas | Section 4 | Transactions: all columns |
| **BQ6** | Is the data ready for reliable analysis, and how different are the insights before and after cleaning? | Preparing data for reliable analysis | Section 5 | Transactions: all columns |
| **BQ7** | Can AI check our cleaning logic and correct our code, and can we trust its answers? | AI support | Section 6 | Transactions: all columns |

## The Dataset: 14 variables
| # | Variable | Meaning | Valid rule |
|---|---|---|---|
| 1 | `Txn_ID` | Transaction number | Unique |
| 2 | `Txn_Date` | Date of sale | 1 Jul 2026 to 30 Sep 2026 |
| 3 | `Store_City` | City of the store | Mumbai, Delhi, Bengaluru, Hyderabad |
| 4 | `Store_Format` | Type of Reliance Retail store | Reliance Fresh, JioMart, Reliance Digital, Trends |
| 5 | `Category` | Product category | Grocery, Electronics, Fashion, Home |
| 6 | `Unit_Price` | Price of ONE item (Rs) | Number above 0 |
| 7 | `Quantity` | Items sold | Whole number above 0 |
| 8 | `Discount_Pct` | Discount given (%) | 0 to 100 |
| 9 | `Payment_Mode` | How the customer paid | UPI, Card, Cash, Wallet |
| 10 | `Customer_Type` | New, Returning or Loyalty | Listed types |
| 11 | `Customer_Age` | Age from the loyalty app | 21 to 65 |
| 12 | `Monthly_Income` | Income told by the customer (Rs) | Number above 0 |
| 13 | `Customer_Rating` | Rating of the purchase | 1 to 5 |
| 14 | `Sales_Amount` | Amount recorded by the store system | `Unit_Price x Quantity x (1 - Discount_Pct/100)` |

## Suggested timing (90 minutes)
| Section | Topic | Time |
|---|---|---|
| Step 0 | Setup, load and quality overview | 5 min |
| 1 | Missing values (MCAR, MAR, MNAR and 5 methods) | 30 min |
| 2 | Duplicate records | 8 min |
| 3 | Inconsistent and incorrect entries | 12 min |
| 4 | Data cleaning using Pandas | 12 min |
| 5 | Preparing data for reliable analysis | 10 min |
| 6 | AI support for cleaning checks and code correction | 8 min |
| 7 | Visualisation questions and practice | 5 min |

## How to use this notebook
1. Upload `Reliance_Retail_Transactions_v2.xlsx` to Colab (📁 icon on the left). **If you uploaded an older copy earlier, delete it first**, otherwise Colab keeps the old file.
2. Run the cells **in order**. Above every code cell you will find the **business question** it answers.
3. Sections 1 to 3 teach **one problem at a time**. Section 4 fixes **everything together in the correct order**.

---
# Step 0: Setup, Load and Quality Overview

💼 **Business question:** What does the raw data from the store systems look like before we touch it?

In [ ]:
import pandas as pd            # tables
import numpy as np             # numbers
import matplotlib.pyplot as plt   # charts
import seaborn as sns          # nicer charts

# Read the main sheet. 'raw' is the original data: we NEVER change it.
raw = pd.read_excel("Reliance_Retail_Transactions_v2.xlsx", sheet_name="Transactions")

print("Rows   :", raw.shape[0])
print("Columns:", raw.shape[1])
raw.head()

💼 **Business question:** Did we load the latest version of the data file? (Old copies cause errors such as KeyError: 'Customer_Age'.)

In [ ]:
needed = ["Txn_ID", "Txn_Date", "Store_City", "Store_Format", "Category", "Unit_Price", "Quantity",
          "Discount_Pct", "Payment_Mode", "Customer_Type", "Customer_Age", "Monthly_Income",
          "Customer_Rating", "Sales_Amount"]

all_good = True
for col in needed:
    if col not in raw.columns:
        print("✘ Missing column:", col)
        all_good = False

if raw.shape != (200, 14):
    print("✘ Expected 200 rows and 14 columns, but found", raw.shape)
    all_good = False

if all_good:
    print("✔ Correct file: 200 rows and 14 columns. The notebook and the Excel file match.")
else:
    print()
    print("FIX: 1) Click the folder icon on the left and delete any old Reliance Excel file.")
    print("     2) Upload Reliance_Retail_Transactions_v2.xlsx again.")
    print("     3) Click Runtime > Restart session, then run the cells again from the top.")

💼 **Business question:** What does each column contain, and what data type does Python use for it?

In [ ]:
raw.info()

📌 **How to read the result:** `Non-Null Count` below 200 means that column has blank cells. `object` or `str` means text. Notice that `Unit_Price` is text, which is a warning sign.

💼 **Business question:** If we trusted the raw data completely, what would the dashboard tell the regional head?

In [ ]:
raw_total = raw["Sales_Amount"].sum()
raw_avg_rating = raw["Customer_Rating"].mean()
raw_city = raw.groupby("Store_City")["Sales_Amount"].sum().sort_values(ascending=False)

print("Total sales (raw): Rs", round(raw_total))
print("Average rating (raw):", round(raw_avg_rating, 2))
print()
print("Sales by city (raw):")
print(raw_city.round(0))

### The five qualities of good data
| Quality | Question to ask | Example of a failure |
|---|---|---|
| **Completeness** | Is anything blank? | Rating missing |
| **Uniqueness** | Is any record repeated? | Same transaction twice |
| **Consistency** | Is the same thing written the same way? | "Mumbai", "mumbai", "Bombay" |
| **Validity** | Does it follow the business rules? | Rating = 9, Quantity = -3 |
| **Accuracy** | Do the numbers agree with each other? | Sales_Amount does not equal price × quantity |

We now check each quality, one section at a time.

---
# Section 1: Missing Values (30 min)

**What you will learn**
1. How to find missing values (count, percentage, chart)
2. **Why** values go missing: **MCAR, MAR and MNAR**, with business examples
3. Five ways to handle them: `dropna()`, **mean**, **median**, **mode** and **"Unknown"**
4. How to **choose** a method: *the right method depends on why the data is missing and on the business context*

**Why it matters for Reliance:** blank ages and incomes can make the "average customer" look younger, poorer or richer than the real customer, and a wrong customer picture leads to wrong offers.

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Missing value** | A cell with no value, shown as `NaN`. | A blank `Customer_Rating` |
| **MCAR** | Missing Completely At Random: the blank has no link to any data. | Tablet crash drops a rating |
| **MAR** | Missing At Random: the blank depends on another column we can see. | Age blank when payment is Cash |
| **MNAR** | Missing Not At Random: the blank depends on the missing value itself. | High earners skip income |
| **`isnull()`** | True where a cell is blank. | `raw.isnull().sum()` |
| **`dropna()`** | Deletes rows that contain blanks. | `cust.dropna()` |
| **`fillna(value)`** | Fills blank cells with a chosen value. | `fillna("Unknown")` |
| **Imputation** | Replacing a blank with an estimated value. | Age filled with the mean |
| **Mean** | Sum of values divided by their count; pulled by extremes. | Average age |
| **Median** | The middle value; not pulled by extremes. | Typical income |
| **Mode** | The most frequent value; used for text. | Most common payment mode |
| **Assumption** | A choice you make and write down. | "Blank city rows are deleted" |

## 1.1 Find the missing values

💼 **Business question:** **BQ1:** Which columns have blank cells, and how many?

In [ ]:
missing = raw.isnull().sum()          # True = blank; sum() counts the blanks in each column
print(missing[missing > 0])           # show only columns that have blanks
print("Total blank cells:", missing.sum())

💼 **Business question:** What percentage of each column is blank? (A small gap in an important column still matters.)

In [ ]:
pct = (raw.isnull().mean() * 100).round(1)     # mean of True/False = share of blanks
print(pct[pct > 0])

💼 **Business question:** Where exactly are the blanks? Are they spread randomly or in clusters?

In [ ]:
sns.heatmap(raw.isnull(), cbar=False, yticklabels=False)
plt.title("Chart 1: Missing values (light lines = blank cells)")
plt.show()

📊 **Visualisation questions: Chart 1** (the chart above; answer in a comment or a markdown cell)

1. Which column has the most blank cells?
2. Can blank cells in Customer_Rating make an average rating wrong? Explain in two lines.

💼 **Business question:** Which columns need our attention first?

In [ ]:
pct[pct > 0].plot(kind="bar", title="Chart 2: Missing values by column (%)")
plt.ylabel("% blank")
plt.xticks(rotation=45)
plt.show()

📊 **Visualisation questions: Chart 2** (the chart above; answer in a comment or a markdown cell)

1. Which column has the highest percentage missing?
2. Which missing column would you fix first for a customer-profile report? Why?

## 1.2 Why are values missing? MCAR, MAR and MNAR
Before fixing a blank, ask: **why is it blank?** The reason decides which method is safe.

| Type | Meaning | Simple business example | Reliance column in our data |
|---|---|---|---|
| **MCAR**<br>Missing Completely At Random | The blank has **no link** to any column, seen or unseen. It is pure bad luck. | A store tablet crashes at random and the rating is not saved. | `Customer_Rating`, `Store_City`, `Customer_Type` |
| **MAR**<br>Missing At Random | The blank depends on **another column we can see**. | Age is missing mostly when the customer pays by **Cash**, because cash shoppers do not scan the loyalty card. | `Customer_Age` (depends on `Payment_Mode`) |
| **MNAR**<br>Missing Not At Random | The blank depends on **the missing value itself**. | **High earners skip** the income question, so the missing incomes are the high ones. | `Monthly_Income` |

> **How safe is it?** MCAR is the safest to fix. MAR needs care. MNAR is the most dangerous: **any average will be wrong, and no simple method fixes it.** You can only **argue** MNAR from business knowledge, you cannot prove it from the data alone.

💼 **Business question:** **MCAR check:** Is the blank rating spread evenly across store formats (random), or concentrated in one place?

In [ ]:
rating_blank = raw["Customer_Rating"].isnull()                     # True where rating is blank
blank_by_format = rating_blank.groupby(raw["Store_Format"]).mean() * 100
print(blank_by_format.round(1))

📌 **How to read the result:** If the percentages are small and look similar across formats, nothing in the data explains the blanks. That points to **MCAR**.

💼 **Business question:** **MAR check:** Does the blank age depend on the payment mode?

In [ ]:
# Tidy the payment spelling first (we study this properly in Section 3)
pay = raw["Payment_Mode"].str.strip().str.title().replace({"Upi": "UPI"})

age_blank = raw["Customer_Age"].isnull()
age_blank_pct = (age_blank.groupby(pay).mean() * 100).round(1)
print(age_blank_pct)

age_blank_pct.plot(kind="bar", title="Chart 3: % of Customer_Age blank, by payment mode")
plt.ylabel("% blank")
plt.xticks(rotation=0)
plt.show()

📊 **Visualisation questions: Chart 3** (the chart above; answer in a comment or a markdown cell)

1. Which payment mode has the highest percentage of blank ages?
2. What business reason could explain it? What does this tell you about the missing type?

📌 **How to read the result:** One payment mode has a **much higher** blank rate than the others. The blank depends on another column we can see, so this is **MAR**.

💼 **Business question:** **MNAR check:** Who is skipping the income question? Compare blank incomes with customer type and store format.

In [ ]:
income_blank = raw["Monthly_Income"].isnull()

print("Customer type of people who gave income (False) vs skipped (True), in %:")
print((pd.crosstab(income_blank, raw["Customer_Type"], normalize="index") * 100).round(0))
print()
print("Store format of people who gave income (False) vs skipped (True), in %:")
print((pd.crosstab(income_blank, raw["Store_Format"], normalize="index") * 100).round(0))

📌 **How to read the result:** The customers who **skipped** the question are mostly **Loyalty** customers who shop at **Reliance Digital** (expensive electronics). These are likely the high earners, so the answered incomes are **too low on average**. This pattern is typical of **MNAR**.

### Quick guide: how to recognise the type
| Question to ask | If yes... |
|---|---|
| Does the blank rate change with another column we can see? | **MAR** |
| Is the blank rate the same everywhere, with no business reason? | **MCAR** |
| Would the blank rate depend on the value itself (rich, unhappy, embarrassed customers skip)? | **MNAR** |

## 1.3 Five ways to handle missing values (sample customer dataset)
We learn the methods on a **small sample of 12 customers** (sheet `Customer_Sample`) so that you can see **every row before and after**.

💼 **Business question:** What does the sample customer dataset look like, and where are the blanks?

In [ ]:
cust = pd.read_excel("Reliance_Retail_Transactions_v2.xlsx", sheet_name="Customer_Sample")
print(cust)
print()
print("Blank cells in each column:")
print(cust.isnull().sum())

### Method 1: Delete rows with `dropna()`

💼 **Business question:** What if we simply delete every customer who has a blank? How many customers are left to analyse?

In [ ]:
# METHOD 1: dropna()  = delete rows that contain blanks

# (a) delete a row if ANY column is blank
m1 = cust.dropna()
print("Customers BEFORE:", len(cust))
print("Customers AFTER (any blank deleted):", len(m1))
print(m1)

# (b) delete a row only if ONE chosen column is blank
m1b = cust.dropna(subset=["Monthly_Income"])
print()
print("Customers AFTER (only blank Monthly_Income deleted):", len(m1b))

| | |
|---|---|
| **When to use** | Only a **very small share** of rows is blank (about 1 to 5%), the blanks are **MCAR**, and the column is **essential** (you cannot guess it). |
| **Limitations** | You lose **whole rows**, including their good data. With many blanks you can lose most of your sample. If the data is MAR or MNAR, the rows that remain are **not typical**, so the results become biased. |
| **Business application** | `Store_City` blank in 3 of 200 sales: you cannot tell which city earned the sale, so deleting those 3 rows is acceptable. |

### Method 2: Mean imputation

💼 **Business question:** Customers with a blank age: can we fill the age with the average age, and does the average change?

In [ ]:
# METHOD 2: mean imputation = fill blanks with the average
m2 = cust.copy()                                      # work on a copy

age_mean = m2["Age"].mean()                           # mean() ignores blanks
print("Average age BEFORE:", round(age_mean, 1), "| blanks:", m2["Age"].isnull().sum())

m2["Age"] = m2["Age"].fillna(age_mean)                # fill every blank with the mean

print("Average age AFTER :", round(m2["Age"].mean(), 1), "| blanks:", m2["Age"].isnull().sum())
print(m2[["Customer_ID", "Age"]])

| | |
|---|---|
| **When to use** | A **numeric** column with **no extreme values**, blanks that are **MCAR** and few, and you need a quick estimate. |
| **Limitations** | The **average stays the same**, but every filled customer looks identical, so the **spread shrinks** (the data looks more similar than it is). A few extreme values pull the mean away from the typical customer. It is wrong for MAR or MNAR data. |
| **Business application** | A few blank ages in a loyalty list: fill with the average age to plan an age-based campaign. |

### Method 3: Median imputation

💼 **Business question:** Monthly income has one very rich customer. Should we fill blanks with the mean income or the median income?

In [ ]:
# METHOD 3: median imputation = fill blanks with the middle value
m3 = cust.copy()

income_mean = m3["Monthly_Income"].mean()
income_median = m3["Monthly_Income"].median()
print("Mean income  : Rs", round(income_mean))         # pulled up by one customer earning Rs 2.5 lakh
print("Median income: Rs", round(income_median))       # the typical customer

m3["Monthly_Income"] = m3["Monthly_Income"].fillna(income_median)

print("Blanks AFTER:", m3["Monthly_Income"].isnull().sum())
print(m3[["Customer_ID", "Customer_Type", "Monthly_Income"]])

📌 **How to read the result:** The mean (Rs 65,000) is much higher than the median (Rs 41,000) because one customer earns Rs 2,50,000. Filling every blank with the **mean** would make unknown customers look richer than they probably are.

| | |
|---|---|
| **When to use** | A **numeric** column that is **skewed** or has **outliers** (income, sales amount, delivery time). Blanks are MCAR or the share is small. |
| **Limitations** | Like the mean, every filled value is the same, so the spread shrinks. If the blanks are **MNAR** (the rich skipped), the median of the people who answered is **still too low**. |
| **Business application** | Fill blank order values or incomes with the median so that a few big buyers do not distort the typical customer. |

### Method 4: Mode imputation

💼 **Business question:** Payment mode is text, so there is no average. Can we fill blanks with the most common payment mode?

In [ ]:
# METHOD 4: mode imputation = fill blanks with the most common value (works for text)
m4 = cust.copy()

print("BEFORE:")
print(m4["Payment_Mode"].value_counts(dropna=False))

most_common = m4["Payment_Mode"].mode()[0]            # mode() gives a list; [0] takes the first
print()
print("Most common payment mode:", most_common)

m4["Payment_Mode"] = m4["Payment_Mode"].fillna(most_common)

print()
print("AFTER:")
print(m4["Payment_Mode"].value_counts(dropna=False))

| | |
|---|---|
| **When to use** | A **category** column (text), **one category clearly dominates**, only a few blanks, MCAR. |
| **Limitations** | It **adds fake rows to the biggest group** and makes it look even bigger. If the blank customers really paid with something else (for example Cash), the result is wrong. |
| **Business application** | `Customer_Type` blank in a few sales: fill with the most common type, "Returning". |

### Method 5: Replace with "Unknown"

💼 **Business question:** What if we do not want to guess at all? Can we keep the rows and mark the blank honestly?

In [ ]:
# METHOD 5: replace with "Unknown" = keep the row, label the blank honestly
m5 = cust.copy()

m5["Payment_Mode"] = m5["Payment_Mode"].fillna("Unknown")

print(m5["Payment_Mode"].value_counts())
print()
print(m5[["Customer_ID", "Payment_Mode"]])

| | |
|---|---|
| **When to use** | A **category** column where guessing is risky, the blanks may be **MAR or MNAR**, or the fact that the answer is missing is itself useful. |
| **Limitations** | "Unknown" is a **new group** that the analysis must handle. It does **not** work for numbers (you cannot average "Unknown"). Too many "Unknown" values make a column weak. |
| **Business application** | `Payment_Mode` blank because of a POS failure: label as "Unknown" instead of pretending they paid by UPI. |

### Summary of the five methods
| Method | Code | Best for | Main risk |
|---|---|---|---|
| 1. Delete rows | `df.dropna()` | Very few MCAR blanks in an essential column | Lose rows; biased if not MCAR |
| 2. Mean | `df["col"].fillna(df["col"].mean())` | Numbers without extremes | Spread shrinks; outliers distort |
| 3. Median | `df["col"].fillna(df["col"].median())` | Skewed numbers (income, sales) | Spread shrinks; wrong for MNAR |
| 4. Mode | `df["col"].fillna(df["col"].mode()[0])` | Text with one dominant value | Inflates the biggest group |
| 5. "Unknown" | `df["col"].fillna("Unknown")` | Text where guessing is risky | New group; not for numbers |

## 1.4 Choosing the method: it depends on WHY it is missing and on the BUSINESS
> **No method is "best".** First ask *why is it missing* (MCAR, MAR or MNAR) and *what will the number be used for*.

| Reliance column | Why missing | Suggested method | Business reason |
|---|---|---|---|
| `Store_City` (3 blanks) | MCAR | **1. Delete rows** | Very few; we cannot guess the city; needed for city reports |
| `Customer_Age` | MAR (Cash payers) | **2. Mean** (first-level choice) | Age is roughly balanced. A better method for later: fill by payment-mode group |
| `Monthly_Income` | MNAR | **3. Median** + a warning | Income is skewed. The result is still **too low**, so report it with a caution |
| `Customer_Rating` | MCAR | **3. Median** | Ratings 1 to 5; the median is a typical rating |
| `Customer_Type` | MCAR | **4. Mode** | "Returning" is the dominant type |
| `Payment_Mode` | MCAR (POS failure) | **5. "Unknown"** | Guessing a payment mode would invent facts |

💼 **Business question:** On the Reliance data, how much do the averages change with each method? (Before and after results.)

In [ ]:
# Customer_Age
print("Average age: blanks ignored          :", round(raw["Customer_Age"].mean(), 1))
print("Average age: after MEAN fill         :", round(raw["Customer_Age"].fillna(raw["Customer_Age"].mean()).mean(), 1))
print()
# Monthly_Income
print("Average income: blanks ignored       : Rs", round(raw["Monthly_Income"].mean()))
print("Average income: after MEDIAN fill    : Rs", round(raw["Monthly_Income"].fillna(raw["Monthly_Income"].median()).mean()))
print()
print("Blank ages   :", raw["Customer_Age"].isnull().sum(), "of", len(raw))
print("Blank incomes:", raw["Monthly_Income"].isnull().sum(), "of", len(raw))

📌 **How to read the result:** Filling the blank incomes with the median **pulls the average income down** (from Rs 45,820 to Rs 45,425) because the filled value is lower than the missing high earners. Since income is **MNAR**, both numbers are probably **too low**. Say this in your report.

## 1.5 Practical business exercise: the Premium Loyalty Offer
**Situation.** The marketing head wants to launch a **Premium Loyalty offer** for customers **aged 30 to 50** who earn **above Rs 60,000 per month**. She asks you for (1) the average age, (2) the average income and (3) how many customers fit the target group. Customer ages and incomes have blanks.

**Your tasks** (use a copy: `ex = raw.copy()`)
1. Count the blanks in `Customer_Age` and `Monthly_Income` and find the percentage missing.
2. For each column, decide the type (**MCAR, MAR or MNAR**) and write one line of evidence.
3. Choose a method for each column from the five methods and give a **business reason**.
4. Apply your methods to `ex`. Show the average age and the average income **before and after**.
5. Count how many customers are aged 30 to 50 **and** earn above Rs 60,000 (use comparisons with `&`).
6. Write **3 lines** for the head: how much can she trust the income number, and why?

**Hints:** `ex["Customer_Age"].isnull().sum()` counts blanks. `ex["Monthly_Income"].fillna(...)` fills blanks. `((ex["Customer_Age"] >= 30) & (ex["Customer_Age"] <= 50) & (ex["Monthly_Income"] > 60000)).sum()` counts the target group.

In [ ]:
# Write your solution here
ex = raw.copy()

### ✅ Key takeaways
- A missing value is a blank cell (`NaN`). Find it with `isnull().sum()`, the percentage, and a heatmap.
- **MCAR** = random, **MAR** = depends on another visible column, **MNAR** = depends on the missing value itself.
- Five methods: `dropna()`, mean, median, mode, "Unknown".
- **The choice depends on why the data is missing and on the business use.** Always write down your assumption.
- Never fill with 0 unless 0 is a true value. For MNAR data, add a warning to your report.

---
# Section 2: Duplicate Records (8 min)

**What you will learn:** how to detect duplicate records, measure the damage, and remove them safely.

**Why it matters for Reliance:** a store system that sends a transaction twice makes **sales look bigger than they are**, and the dashboard no longer matches finance.

| Type | Meaning | Action |
|---|---|---|
| **Exact duplicate** | Every column is the same | Remove the extra copies |
| **Same ID, different values** | Same `Txn_ID` but other columns differ | **Investigate** first; do not just delete |

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Duplicate record** | The same record stored more than once. | A transaction sent twice |
| **`duplicated()`** | True for each repeated row after its first appearance. | `raw.duplicated().sum()` |
| **`duplicated(keep=False)`** | Marks every copy, including the first. | Shows both copies of a pair |
| **`drop_duplicates()`** | Returns the data without repeated rows. | `raw.drop_duplicates()` |
| **Key column** | A column that must be unique for every row. | `Txn_ID` |
| **Double counting** | Adding the same sale twice, which overstates totals. | Dashboard higher than finance |

💼 **Business question:** **BQ2:** Are any transactions recorded twice?

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())          # counts every extra copy
print("Is Txn_ID unique?", raw["Txn_ID"].is_unique)       # a key column must be unique

💼 **Business question:** Which transactions are repeated? (Look at each pair side by side.)

In [ ]:
both = raw[raw.duplicated(keep=False)]                    # keep=False marks BOTH copies
both = both.sort_values("Txn_ID")
print(both[["Txn_ID", "Txn_Date", "Store_City", "Category", "Quantity", "Sales_Amount"]])

💼 **Business question:** Is any transaction ID repeated with DIFFERENT values? (That would need investigation.)

In [ ]:
no_copies = raw.drop_duplicates()
print("Repeated IDs after removing exact copies:", no_copies["Txn_ID"].duplicated().sum())

📌 **How to read the result:** A result of **0** means all repeated IDs were exact copies, so it is safe to remove them.

💼 **Business question:** How much sales amount is counted twice?

In [ ]:
dup_sales = raw[raw.duplicated()]["Sales_Amount"].sum()
print("Sales counted twice: Rs", round(dup_sales))
print("Share of total sales:", round(dup_sales / raw["Sales_Amount"].sum() * 100, 1), "%")

💼 **Business question:** Show the effect of duplicates on the total sales in one picture.

In [ ]:
compare = pd.Series({"With duplicates": raw["Sales_Amount"].sum(),
                     "Without duplicates": raw.drop_duplicates()["Sales_Amount"].sum()})
compare.plot(kind="bar", title="Chart 4: Total sales with vs without duplicates")
plt.ylabel("Sales (Rs)")
plt.xticks(rotation=0)
plt.show()

📊 **Visualisation questions: Chart 4** (the chart above; answer in a comment or a markdown cell)

1. By how much does the total fall when duplicates are removed?
2. Why might the finance department's number be lower than the dashboard's?

💼 **Business question:** How do we remove duplicates and prove that it worked?

In [ ]:
dup_free = raw.drop_duplicates()          # keeps the FIRST copy, removes the rest

print("Rows before:", len(raw))
print("Rows after :", len(dup_free))
print("Txn_ID unique now?", dup_free["Txn_ID"].is_unique)

| Option | Code | Meaning |
|---|---|---|
| Keep the first copy (default) | `drop_duplicates()` | Remove later copies |
| Keep the last copy | `drop_duplicates(keep="last")` | Remove earlier copies |
| Decide by one column | `drop_duplicates(subset="Txn_ID")` | Rows with the same ID count as duplicates |

### ✅ Key takeaways
- `duplicated().sum()` counts repeated rows; `duplicated(keep=False)` shows every copy.
- Always measure the **sales amount** counted twice, not only the number of rows.
- Remove with `drop_duplicates()` and **prove it** with `is_unique`.
- Same ID with different values is not a simple duplicate: investigate it.

---
# Section 3: Inconsistent and Incorrect Entries (12 min)

**What you will learn:** how to detect and handle two different problems.

| Problem | Meaning | Reliance example |
|---|---|---|
| **3A. Inconsistent entries** | The same thing is written in different ways | `Mumbai`, `mumbai`, `MUMBAI `, `Bombay` |
| **3B. Incorrect (invalid) entries** | A value breaks a business rule | Quantity = -3, Rating = 9, Unit_Price = 0 |

**Why it matters for Reliance:** the dashboard counts "Mumbai" and "Bombay" as **two cities**, so each looks smaller than it is.

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Inconsistent entry** | The same value written in different ways. | mumbai, MUMBAI, Bombay |
| **Invalid (incorrect) entry** | A value that breaks a business rule. | Rating = 9 |
| **Outlier** | A value far away from the others; may be real or an error. | Quantity = 500 |
| **Logic check** | A check between columns. | Sales_Amount vs price × quantity |
| **`str.strip()`** | Removes spaces at the start and end of text. | `"MUMBAI "` becomes `"MUMBAI"` |
| **`str.title()`** | Writes Each Word With A Capital Letter. | `"delhi"` becomes `"Delhi"` |
| **`replace()`** | Swaps one value for another. | Bombay to Mumbai |
| **Data type** | How Python stores a column: number, text, date. | Unit_Price stored as text |

## 3A. Detect and handle inconsistent entries

💼 **Business question:** **BQ3:** Is each city written in only one way?

In [ ]:
print(raw["Store_City"].value_counts(dropna=False))      # dropna=False also shows the blanks

💼 **Business question:** Show how many different city names the dashboard sees.

In [ ]:
sns.countplot(data=raw, y="Store_City")
plt.title("Chart 5: Transactions by city name (raw data)")
plt.show()

📊 **Visualisation questions: Chart 5** (the chart above; answer in a comment or a markdown cell)

1. How many different city names does the chart show? How many real cities are there?
2. How does this spelling problem affect a city-wise sales report?

💼 **Business question:** Are the category and payment mode columns also written in different ways?

In [ ]:
print("Category values    :", list(raw["Category"].unique()))
print("Payment_Mode values:", list(raw["Payment_Mode"].unique()))

💼 **Business question:** **Handle it:** How do we make city names consistent? (Three small steps, on a copy.)

In [ ]:
text_demo = raw.copy()                                    # practise on a copy

print("City names BEFORE:", text_demo["Store_City"].nunique())

# Step 1: remove extra spaces at the start and end
text_demo["Store_City"] = text_demo["Store_City"].str.strip()

# Step 2: write every name in Title Case (first letter capital)
text_demo["Store_City"] = text_demo["Store_City"].str.title()

# Step 3: replace different names for the same city
text_demo["Store_City"] = text_demo["Store_City"].replace({"Bombay": "Mumbai",
                                                           "New Delhi": "Delhi",
                                                           "Bangalore": "Bengaluru"})

print("City names AFTER :", text_demo["Store_City"].nunique())
print(text_demo["Store_City"].value_counts(dropna=False))

💼 **Business question:** Does the same recipe fix the category and payment mode?

In [ ]:
text_demo["Category"] = text_demo["Category"].str.strip().str.title()
text_demo["Category"] = text_demo["Category"].replace({"Groceries": "Grocery"})

text_demo["Payment_Mode"] = text_demo["Payment_Mode"].str.strip().str.title()
text_demo["Payment_Mode"] = text_demo["Payment_Mode"].replace({"Upi": "UPI"})     # title() turns UPI into Upi

print(text_demo["Category"].value_counts())
print(text_demo["Payment_Mode"].value_counts(dropna=False))

## 3B. Detect and handle incorrect (invalid) entries
Every column has a **business rule** (see the Data Dictionary). A value is **invalid** when it breaks the rule.

| Rule | Valid | How we check |
|---|---|---|
| Quantity | above 0 and not unrealistic (say 50 or less) | `<= 0`, `> 50` |
| Unit_Price | above 0 | `== 0` |
| Customer_Rating | 1 to 5 | `< 1`, `> 5` |
| Discount_Pct | 0 to 100 | `> 100` |
| Txn_Date | inside the quarter | `> "2026-09-30"` |
| Sales_Amount | equals price × quantity × (1 - discount/100) | logic check |

💼 **Business question:** **BQ4:** Which values are impossible? (Check every business rule with a comparison.)

In [ ]:
print("Quantity 0 or less        :", (raw["Quantity"] <= 0).sum())
print("Quantity above 50 (typo?) :", (raw["Quantity"] > 50).sum())
print("Unit_Price equal to 0     :", (raw["Unit_Price"] == 0).sum())
print("Rating below 1 or above 5 :", ((raw["Customer_Rating"] < 1) | (raw["Customer_Rating"] > 5)).sum())
print("Discount above 100        :", (raw["Discount_Pct"] > 100).sum())
print("Date after 30-Sep-2026    :", (raw["Txn_Date"] > "2026-09-30").sum())

💼 **Business question:** Look at the actual rows. Which transactions have a wrong rating or quantity?

In [ ]:
bad_rating = raw[(raw["Customer_Rating"] < 1) | (raw["Customer_Rating"] > 5)]
print(bad_rating[["Txn_ID", "Customer_Rating"]])
print()
bad_qty = raw[(raw["Quantity"] <= 0) | (raw["Quantity"] > 50)]
print(bad_qty[["Txn_ID", "Category", "Unit_Price", "Quantity", "Sales_Amount"]])

💼 **Business question:** Is Unit_Price really a number in every row? (Check the data type, then look for text.)

In [ ]:
print("Data type of Unit_Price:", raw["Unit_Price"].dtype)

text_price = raw[raw["Unit_Price"].astype(str).str.contains("Rs")]      # rows where the price contains "Rs"
print(text_price[["Txn_ID", "Unit_Price", "Sales_Amount"]])

💼 **Business question:** Is there a quantity that looks like a typing mistake?

In [ ]:
sns.boxplot(data=raw, x="Quantity")
plt.title("Chart 6: Quantity per transaction (raw data)")
plt.show()

📊 **Visualisation questions: Chart 6** (the chart above; answer in a comment or a markdown cell)

1. What does the dot far away from the box mean?
2. Should we delete or correct it? How will you decide?

💼 **Business question:** Which transactions have impossible price and quantity combinations?

In [ ]:
price_number = pd.to_numeric(raw["Unit_Price"], errors="coerce")     # text prices become blank, only for this chart

plt.scatter(price_number, raw["Quantity"])
plt.title("Chart 7: Unit_Price vs Quantity (raw data)")
plt.xlabel("Unit price (Rs)")
plt.ylabel("Quantity")
plt.show()

📊 **Visualisation questions: Chart 7** (the chart above; answer in a comment or a markdown cell)

1. Which dots look impossible, and why?
2. Which rule (Quantity above 0, Unit_Price above 0) catches each one?

💼 **Business question:** **Logic check:** Does the recorded Sales_Amount equal Unit_Price × Quantity × (1 - Discount/100)?

In [ ]:
expected = price_number * raw["Quantity"] * (1 - raw["Discount_Pct"] / 100)
difference = (raw["Sales_Amount"] - expected).abs()

print("Rows where the recorded amount does not match:", (difference > 1).sum())
print(raw[difference > 1][["Txn_ID", "Unit_Price", "Quantity", "Discount_Pct", "Sales_Amount"]])

### How to handle each incorrect value
| Problem | Handling choice | Reason |
|---|---|---|
| Quantity 0 or less, Quantity above 50, Unit_Price 0 | **Delete the row** | Cannot be a real sale |
| Rating outside 1 to 5 | **Set to blank**, then fill it (Section 1 methods) | The customer did rate, but we do not know the value |
| Discount above 100 | **Set to 0** (write the assumption down) | A typing error; the true discount is unknown |
| Date outside the quarter | **Delete the row** | We cannot be sure of the right date |
| Price typed as text ("Rs 450") | **Convert to a number** | The information is correct, only the format is wrong |
| Recorded amount does not match | **Recalculate** it from price, quantity and discount | The formula is the trusted source |

### ✅ Key takeaways
- **Inconsistent** = same thing, different spelling. Fix with `str.strip()`, `str.title()` and `replace()`.
- **Incorrect** = breaks a business rule. Detect with comparisons (`<=`, `>`, `==`) and charts.
- Choose between **delete, set blank, correct or recalculate**, and write down every assumption.
- Check the **data type** of every column: numbers stored as text break calculations.

---
# Section 4: Data Cleaning using Pandas (12 min)

**What you will learn:** how to apply **all** the fixes together, in the correct order, on a **copy** of the data, and keep a **cleaning log**.

**The order matters:**
```
1 duplicates → 2 text spelling → 3 impossible values → 4 missing values → 5 recalculate amounts → 6 cleaning log
```
For example, impossible ratings must be turned into blanks **before** we fill the blanks, and rows must be deleted **before** we calculate means and medians.

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Data cleaning** | Finding and fixing errors so the data can be trusted. | The `clean` table |
| **`copy()`** | Makes a separate copy so the original stays safe. | `clean = raw.copy()` |
| **Boolean filter** | Keeps only rows where a condition is True. | `clean[clean["Quantity"] > 0]` |
| **`loc[condition, column]`** | Changes a column only in the rows that meet a condition. | Set bad ratings to blank |
| **Order of cleaning** | Steps must run in a sensible order. | Bad rating to blank, then fill |
| **Cleaning log** | A table of issues found and actions taken. | The `log` table |
| **Derived column** | A new column calculated from others. | `Net_Amount` |

💼 **Business question:** **BQ5, Step 1:** Remove double counting. How many rows remain?

In [ ]:
clean = raw.copy()                       # clean is a COPY: raw stays safe
print("Rows before:", len(clean))

clean = clean.drop_duplicates()
print("Rows after removing duplicates:", len(clean))

💼 **Business question:** **Step 2:** Make city, category, payment mode and customer type consistent.

In [ ]:
for col in ["Store_City", "Category", "Payment_Mode", "Customer_Type"]:
    clean[col] = clean[col].str.strip().str.title()          # remove spaces, use Title Case

clean["Payment_Mode"] = clean["Payment_Mode"].replace({"Upi": "UPI"})
clean["Store_City"] = clean["Store_City"].replace({"Bombay": "Mumbai", "New Delhi": "Delhi", "Bangalore": "Bengaluru"})
clean["Category"] = clean["Category"].replace({"Groceries": "Grocery"})

print(clean["Store_City"].value_counts(dropna=False))

💼 **Business question:** **Step 3a:** Turn the text prices (such as 'Rs 450') into numbers.

In [ ]:
clean["Unit_Price"] = clean["Unit_Price"].astype(str).str.replace("Rs", "").str.strip().astype(float)
print("Data type of Unit_Price now:", clean["Unit_Price"].dtype)

💼 **Business question:** **Step 3b:** Delete the rows that cannot be real sales (impossible quantity, zero price, date outside the quarter).

In [ ]:
rows_before = len(clean)

clean = clean[(clean["Quantity"] > 0) &
              (clean["Quantity"] <= 50) &
              (clean["Unit_Price"] > 0) &
              (clean["Txn_Date"] <= "2026-09-30")].copy()

print("Rows deleted:", rows_before - len(clean), "| Rows left:", len(clean))

💼 **Business question:** **Step 3c:** Fix a wrong rating and a wrong discount without deleting the sale.

In [ ]:
# Rating outside 1-5 becomes blank (we fill it in Step 4)
clean.loc[(clean["Customer_Rating"] < 1) | (clean["Customer_Rating"] > 5), "Customer_Rating"] = np.nan

# Discount above 100 is a typing error: assume no discount (assumption written down)
clean.loc[clean["Discount_Pct"] > 100, "Discount_Pct"] = 0

📌 **How to read the result:** `clean.loc[condition, column] = value` means: *in the rows where the condition is true, change this column to this value.*

💼 **Business question:** **Step 4:** Handle the missing values. Use the method we chose for each column in Section 1.4.

In [ ]:
print("Blank cells BEFORE:", clean.isnull().sum().sum())

# Method 1: delete rows (very few blanks, essential column)
clean = clean.dropna(subset=["Store_City"]).copy()

# Method 2: mean imputation (age, numeric, no extremes)
clean["Customer_Age"] = clean["Customer_Age"].fillna(clean["Customer_Age"].mean())

# Method 3: median imputation (income and rating, skewed or ordered numbers)
clean["Monthly_Income"] = clean["Monthly_Income"].fillna(clean["Monthly_Income"].median())
clean["Customer_Rating"] = clean["Customer_Rating"].fillna(clean["Customer_Rating"].median())

# Method 4: mode imputation (customer type, one dominant value)
clean["Customer_Type"] = clean["Customer_Type"].fillna(clean["Customer_Type"].mode()[0])

# Method 5: "Unknown" (payment mode, guessing would invent facts)
clean["Payment_Mode"] = clean["Payment_Mode"].fillna("Unknown")

print("Blank cells AFTER :", clean.isnull().sum().sum())

💼 **Business question:** **Step 5:** Can we trust the recorded Sales_Amount? Recalculate it and compare.

In [ ]:
clean["Net_Amount"] = clean["Unit_Price"] * clean["Quantity"] * (1 - clean["Discount_Pct"] / 100)

difference = (clean["Sales_Amount"] - clean["Net_Amount"]).abs()
print("Rows where the recorded amount did not match:", (difference > 1).sum())
print(clean[difference > 1][["Txn_ID", "Unit_Price", "Quantity", "Discount_Pct", "Sales_Amount", "Net_Amount"]])
print()
print("From now on we use Net_Amount as the reliable sales figure.")

💼 **Business question:** **Step 6:** Can we show the head a simple record of what was wrong and what we did?

In [ ]:
log = pd.DataFrame({
    "Issue": ["Duplicate rows",
              "Different city names (should be 4)",
              "Price typed as text",
              "Impossible quantity or zero price (rows)",
              "Date after 30-Sep-2026",
              "Rating outside 1 to 5",
              "Discount above 100",
              "Blank cells (all columns)"],
    "Found in raw data": [raw.duplicated().sum(),
                          raw["Store_City"].nunique(),
                          raw["Unit_Price"].astype(str).str.contains("Rs").sum(),
                          ((raw["Quantity"] <= 0) | (raw["Quantity"] > 50) | (raw["Unit_Price"] == 0)).sum(),
                          (raw["Txn_Date"] > "2026-09-30").sum(),
                          ((raw["Customer_Rating"] < 1) | (raw["Customer_Rating"] > 5)).sum(),
                          (raw["Discount_Pct"] > 100).sum(),
                          raw.isnull().sum().sum()],
    "Action": ["Removed", "Standardised spelling", "Converted to number", "Deleted rows", "Deleted rows",
               "Set blank, then median fill", "Set to 0 (assumption)", "Deleted / mean / median / mode / Unknown"],
})
log

### ✅ Key takeaways
- Always clean a **copy** (`raw.copy()`); never change the raw data.
- Follow the **order**: duplicates, spelling, impossible values, missing values, recalculate.
- Use `loc[condition, column] = value` to fix some rows only.
- Keep a **cleaning log**: issue, how many, what you did.

---
# Section 5: Preparing Data for Reliable Analysis (10 min)

**What you will learn:** how to **prove** the data is ready, add helper columns, save it, and **compare insights before and after cleaning** (this answers the central question).

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Validation check** | A test that confirms a rule is now true. | `(clean["Quantity"] > 0).all()` |
| **`all()`** | True only if every value is True. | All quantities positive |
| **`between(a, b)`** | True when a value is from a to b. | Rating between 1 and 5 |
| **Reliability of insights** | How far a decision based on the numbers can be trusted. | City ranking before vs after |
| **Overstated / under-reported** | Reported higher / lower than the truth. | Total sales / Mumbai |
| **`to_excel()`** | Saves a table as an Excel file. | `Reliance_Retail_Clean.xlsx` |

💼 **Business question:** **BQ6:** Does the cleaned data now follow every business rule? (Validation checks.)

In [ ]:
if clean.isnull().sum().sum() == 0:
    print("✔ No blank cells")
else:
    print("✘ Blank cells remain")

if clean.duplicated().sum() == 0 and clean["Txn_ID"].is_unique:
    print("✔ No duplicates, Txn_ID is unique")
else:
    print("✘ Duplicates remain")

if (clean["Quantity"] > 0).all() and (clean["Unit_Price"] > 0).all():
    print("✔ Quantity and price are positive")
else:
    print("✘ Impossible quantity or price remains")

if clean["Customer_Rating"].between(1, 5).all() and clean["Discount_Pct"].between(0, 100).all():
    print("✔ Ratings are 1-5 and discounts are 0-100")
else:
    print("✘ Invalid rating or discount remains")

if clean["Txn_Date"].max() <= pd.Timestamp("2026-09-30"):
    print("✔ All dates are inside the quarter")
else:
    print("✘ Some dates are outside the quarter")

print("City names now:", sorted(clean["Store_City"].unique()))

💼 **Business question:** How much better is the data quality now? (A simple scorecard.)

In [ ]:
print("                      RAW      CLEAN")
print("Rows                :", len(raw), "    ", len(clean))
print("Blank cells         :", raw.isnull().sum().sum(), "      ", clean.isnull().sum().sum())
print("Duplicate rows      :", raw.duplicated().sum(), "      ", clean.duplicated().sum())
print("Different city names:", raw["Store_City"].nunique(), "     ", clean["Store_City"].nunique())

💼 **Business question:** Prepare the data for analysis: add a Month column and check the data types.

In [ ]:
clean["Month"] = clean["Txn_Date"].dt.month_name()      # July, August, September
print(clean["Month"].value_counts())
print()
print(clean[["Txn_Date", "Unit_Price", "Quantity", "Customer_Age", "Monthly_Income", "Net_Amount"]].dtypes)

### Before vs after: how reliable were the insights? (the central question)

💼 **Business question:** By how much did the raw data overstate total sales?

In [ ]:
clean_total = clean["Net_Amount"].sum()

print("Total sales, raw data  : Rs", round(raw_total))
print("Total sales, clean data: Rs", round(clean_total))
print("Overstated by          : Rs", round(raw_total - clean_total))
print("Overstated by          :", round((raw_total / clean_total - 1) * 100, 1), "%")

💼 **Business question:** Does the city ranking change? Which city was under-reported because its name was split into pieces?

In [ ]:
before = raw.groupby("Store_City")["Sales_Amount"].sum()
after = clean.groupby("Store_City")["Net_Amount"].sum()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
before.sort_values().plot(kind="barh", ax=ax[0], title="Chart 8a: BEFORE cleaning (11 city names)")
after.sort_values().plot(kind="barh", ax=ax[1], title="Chart 8b: AFTER cleaning (4 city names)")
ax[0].set_xlabel("Sales (Rs)")
ax[1].set_xlabel("Sales (Rs)")
plt.show()

print("Mumbai before:", round(before["Mumbai"]), "| Mumbai after:", round(after["Mumbai"]))
print("Top city before:", before.idxmax(), "| Top city after:", after.idxmax())

📊 **Visualisation questions: Chart 8** (the chart above; answer in a comment or a markdown cell)

1. Which city moves up the most after cleaning?
2. Which decision would have gone wrong if the head had used the 'before' chart?

💼 **Business question:** Why does the shape of the order-value histogram change?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(raw["Sales_Amount"], bins=20)
ax[0].set_title("Chart 9a: BEFORE: Sales_Amount (raw)")
ax[1].hist(clean["Net_Amount"], bins=20)
ax[1].set_title("Chart 9b: AFTER: Net_Amount (clean)")
ax[0].set_xlabel("Rs")
ax[1].set_xlabel("Rs")
plt.show()

📊 **Visualisation questions: Chart 9** (the chart above; answer in a comment or a markdown cell)

1. Where is the extreme value in the raw histogram?
2. Why do two histograms of the same business look so different?

💼 **Business question:** Did the average rating, age and income change after handling the blanks?

In [ ]:
print("Average rating, raw  :", round(raw["Customer_Rating"].mean(), 2), "| ratings used:", raw["Customer_Rating"].count())
print("Average rating, clean:", round(clean["Customer_Rating"].mean(), 2), "| ratings used:", clean["Customer_Rating"].count())
print()
print("Average age,    raw  :", round(raw["Customer_Age"].mean(), 1), "| clean:", round(clean["Customer_Age"].mean(), 1))
print("Average income, raw  : Rs", round(raw["Monthly_Income"].mean()), "| clean: Rs", round(clean["Monthly_Income"].mean()))
print()
print("Remember: income is MNAR, so both income numbers are probably too low.")

💼 **Business question:** What share of payments is unknown after cleaning, and what does it say about the data?

In [ ]:
share = clean["Payment_Mode"].value_counts()
plt.pie(share.values, labels=share.index, autopct="%1.0f%%")
plt.title("Chart 10: Payment mode share (clean data)")
plt.show()

📊 **Visualisation questions: Chart 10** (the chart above; answer in a comment or a markdown cell)

1. What percentage of transactions is paid by UPI?
2. Is a pie chart the best choice here? What would you use instead?

💼 **Business question:** Which headline numbers changed, and which decisions would change with them?

In [ ]:
table = pd.DataFrame({
    "Raw dashboard": [round(raw_total), before.idxmax(), round(before["Mumbai"]), round(raw["Customer_Rating"].mean(), 2), len(raw)],
    "After cleaning": [round(clean_total), after.idxmax(), round(after["Mumbai"]), round(clean["Customer_Rating"].mean(), 2), len(clean)],
}, index=["Total sales (Rs)", "Top city", "Mumbai sales (Rs)", "Average rating", "Transactions"])
table

💼 **Business question:** Save the clean data so that others can use it.

In [ ]:
clean.to_excel("Reliance_Retail_Clean.xlsx", index=False)
print("Saved: Reliance_Retail_Clean.xlsx")

### ✅ Key takeaways
- Cleaning is finished only when the data **passes validation checks**.
- The raw dashboard overstated sales by about **40%**, and under-reported Mumbai because its name was split.
- Blank ages and incomes change the customer picture; **MNAR income stays unreliable**, so add a caution.
- Always compare **before and after** to show how data quality changes decisions.

---
# Section 6: AI Support for Cleaning Logic Checks and Code Correction (8 min)

> **Rule: AI helps you check and write. Python proves what is true.**

| AI is good at | AI can go wrong by |
|---|---|
| Spotting steps you forgot (for example, "strip spaces before replacing") | Inventing numbers or counts |
| Explaining an error message | Fixing the wrong line or hiding the real problem |
| Suggesting cleaner code | Using a method that does not exist |
| Checking whether your missing-value choice fits the reason for the blank | Treating **MNAR** data as if it were safe to fill |

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Prompt** | The instruction text given to an AI. | The cleaning-plan review prompt |
| **Hallucination** | When AI states something false that sounds sure. | "6 duplicates" |
| **Verification** | Checking an AI statement with Python. | `claim == actual` |
| **Traceback** | The error message Python prints. | Read it from the bottom |
| **Human in the loop** | A person approves AI output before use. | The analyst |

💼 **Business question:** How do I ask AI to review my cleaning logic?

In [ ]:
steps = '''
1. Remove duplicate rows
2. Strip spaces and use Title Case for city, category, payment mode
3. Replace Bombay with Mumbai, New Delhi with Delhi, Bangalore with Bengaluru
4. Convert Unit_Price from text to number
5. Delete rows where Quantity <= 0 or Quantity > 50 or Unit_Price = 0
6. Delete rows where Store_City is blank
7. Fill blank Customer_Age with the mean, blank Monthly_Income with the median
8. Fill blank Payment_Mode with "Unknown"
'''

prompt = f'''
ROLE: You are a senior data analyst reviewing a junior analyst's cleaning plan.
CONTEXT: Retail transactions of a large company. Customer_Age is blank mostly for Cash payers (MAR).
Monthly_Income is blank mostly for high earners (MNAR).
PLAN:
{steps}
TASK: 1) Tell me if the ORDER of the steps is correct. 2) List any step I forgot.
3) Say if my missing-value choices are safe for MAR and MNAR data.
RULES: Do not invent numbers. Explain each point in one line.
'''
print(prompt)

💼 **Business question:** **Code correction:** This cleaning code has 4 errors. Find them with the error messages and fix them. (After fixing, `Rows left` should be 188.)

In [ ]:
test = raw.copy()
test = test.drop_duplicate()
test["Store_City"] = test["Store_City"].str.strip.title()
test = test[test["Quantity"] > 0]
test["Customer_Rating"] = test["Customer_Rating"].fillna(test["Customer_rating"].median())
print("Rows left:", len(test)

Write a comment above every line you fix:
```python
# ERROR 1 | Type: ... | Wrong code: ... | Reason: ... | FIX: ...
```

💼 **Business question:** AI wrote a summary of our cleaning. Which of its statements are correct? (Verify each claim with Python.)

In [ ]:
ai_draft = '''
AI DRAFT SUMMARY (to be verified)
1. The raw data has 6 duplicate rows.
2. Customer_Rating has 14 missing values.
3. After cleaning, Delhi has the highest sales.
4. All quantities in the raw data are positive.
5. The raw dashboard overstates sales by about 40%.
6. Cleaning removed 20 rows in total.
7. Electronics is the top category after cleaning.
8. The blank incomes are caused by high earners skipping the question.
'''
print(ai_draft)

In [ ]:
# Compare each AI claim with what Python finds
claim = 6
actual = raw.duplicated().sum()
print("Claim 1 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = 14
actual = raw["Customer_Rating"].isnull().sum()
print("Claim 2 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = "Delhi"
actual = clean.groupby("Store_City")["Net_Amount"].sum().idxmax()
print("Claim 3 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = 0
actual = (raw["Quantity"] <= 0).sum()
print("Claim 4 | AI said:", claim, "non-positive | Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = 40
actual = round((raw_total / clean_total - 1) * 100)
print("Claim 5 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = 20
actual = len(raw) - len(clean)
print("Claim 6 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

claim = "Electronics"
actual = clean.groupby("Category")["Net_Amount"].sum().idxmax()
print("Claim 7 | AI said:", claim, "| Python:", actual, "->", "✔ correct" if claim == actual else "✘ wrong")

print()
print("? CANNOT PROVE | Claim 8: the data shows that Loyalty customers skip the income question,")
print("  but WHY they skip (MNAR) can only be argued from business knowledge, not proved with Python.")

### Scorecard (for class discussion)
| Claim | Result |
|---|---|
| 1. 6 duplicates | ✘ Wrong number (there are 10) |
| 2. 14 missing ratings | ✔ Correct |
| 3. Delhi is top | ✘ Wrong ranking (Mumbai is top) |
| 4. All quantities positive | ✘ Invented fact (2 are 0 or less) |
| 5. About 40% overstated | ✔ Correct |
| 6. 20 rows removed | ✔ Correct |
| 7. Electronics top category | ✔ Correct |
| 8. High earners skip income | ? Cannot be proved with this data |

### ✅ Key takeaways
- Ask AI to **review the logic and the order** of your steps, then test the code yourself.
- Read the error message from the **bottom**: it names the problem and the line.
- Verify **every number** an AI gives with Python before you use it.

---
# Section 7: Visualisation Questions and Practice (5 min)
Seven chart types, five question types each: **Read, Compare, Interpret, Decide, Fix / Choose.** The Word file contains the same 35 questions.

| No. | Chart | Where | Type | Question |
|---|---|---|---|---|
| VQ1.1 | Heatmap | Chart 1 (Section 1) | Read | Which column has the most blank cells? Confirm the count with isnull().sum(). |
| VQ1.2 | Heatmap | Chart 1 (Section 1) | Compare | Compare Customer_Rating and Customer_Age: which has more blank cells, and by how many? |
| VQ1.3 | Heatmap | Chart 1 (Section 1) | Interpret | Can blank cells in Customer_Rating make an average rating wrong? Explain in two lines. |
| VQ1.4 | Heatmap | Chart 1 (Section 1) | Decide | A colleague says 'delete every row that has any blank cell'. Check raw.dropna().shape. How many rows would you lose, and is it a good decision? |
| VQ1.5 | Heatmap | Chart 1 (Section 1) | Fix / Choose | Change the heatmap to show only the first 30 rows, with a title. Why is a bar chart of missing % easier for a manager to read? |
| VQ2.1 | Bar chart | Charts 4 and 8 (Sections 2 and 5) | Read | In the 'before' chart, what is the sales figure of the label 'Mumbai'? What is it in the 'after' chart? |
| VQ2.2 | Bar chart | Charts 4 and 8 (Sections 2 and 5) | Compare | Which city moves up the most in the ranking after cleaning? By how much does its sales figure change? |
| VQ2.3 | Bar chart | Charts 4 and 8 (Sections 2 and 5) | Interpret | Why does the 'before' chart show 11 bars and the 'after' chart only 4? |
| VQ2.4 | Bar chart | Charts 4 and 8 (Sections 2 and 5) | Decide | If marketing money had been shared using the 'before' chart, which city would have been under-funded, and why? |
| VQ2.5 | Bar chart | Charts 4 and 8 (Sections 2 and 5) | Fix / Choose | Edit the 'after' chart: sort the bars, label the axis with Rs, and start the axis at 0. Why must a bar chart start at 0? |
| VQ3.1 | Count plot | Chart 5 (Section 3A) | Read | How many different city names does the count plot show? How many real cities are there? |
| VQ3.2 | Count plot | Chart 5 (Section 3A) | Compare | Which spelling of Mumbai has the most rows: 'Mumbai', 'mumbai', 'MUMBAI ' or 'Bombay'? |
| VQ3.3 | Count plot | Chart 5 (Section 3A) | Interpret | Which type of data-quality problem does this chart show: missing, duplicate, inconsistent or incorrect? Explain. |
| VQ3.4 | Count plot | Chart 5 (Section 3A) | Decide | What rule would you give store managers so that city names are always entered in one way? |
| VQ3.5 | Count plot | Chart 5 (Section 3A) | Fix / Choose | Redraw the count plot after cleaning. What changes? Use order=... to show the cities from highest to lowest. |
| VQ4.1 | Boxplot | Chart 6 (Section 3B) | Read | What is the largest Quantity in the data? Is it visible on the boxplot? |
| VQ4.2 | Boxplot | Chart 6 (Section 3B) | Compare | Compare the box before and after removing the outlier. Which version is more useful for a store manager? |
| VQ4.3 | Boxplot | Chart 6 (Section 3B) | Interpret | What does the dot far away from the box mean? Is it a real bulk order or a typing error? What evidence do you have? |
| VQ4.4 | Boxplot | Chart 6 (Section 3B) | Decide | Write a rule for the maximum allowed Quantity and justify it with numbers from the data. |
| VQ4.5 | Boxplot | Chart 6 (Section 3B) | Fix / Choose | Zoom the boxplot to Quantity 0 to 10 with the axis limit. What do you lose? When can zooming mislead? |
| VQ5.1 | Scatter plot | Chart 7 (Section 3B) | Read | How many dots have Quantity below 1, or Unit_Price equal to 0? |
| VQ5.2 | Scatter plot | Chart 7 (Section 3B) | Compare | Colour the dots by Category using hue. Which category has high prices but low quantities? |
| VQ5.3 | Scatter plot | Chart 7 (Section 3B) | Interpret | Which dots look impossible, and which validity rule catches each one? |
| VQ5.4 | Scatter plot | Chart 7 (Section 3B) | Decide | Should a transaction with Unit_Price 0 be deleted or investigated? Give one reason for each choice. |
| VQ5.5 | Scatter plot | Chart 7 (Section 3B) | Fix / Choose | Add axis titles with units and use a log scale on the X axis. Why does a log scale help when prices range from Rs 50 to Rs 60,000? |
| VQ6.1 | Histogram | Chart 9 (Section 5) | Read | In which Rs range do most transactions fall in the clean histogram? |
| VQ6.2 | Histogram | Chart 9 (Section 5) | Compare | Compare the X axis range of the raw and the clean histogram. What causes the difference? |
| VQ6.3 | Histogram | Chart 9 (Section 5) | Interpret | Why does one wrong value (500 items) change the whole shape of the raw histogram? |
| VQ6.4 | Histogram | Chart 9 (Section 5) | Decide | Which number would you report to the regional head: the mean or the median order value? Why? |
| VQ6.5 | Histogram | Chart 9 (Section 5) | Fix / Choose | Change the number of bins to 10 and then to 40. Which one shows the shape best? Add axis titles. |
| VQ7.1 | Pie chart | Chart 10 (Section 5) | Read | What percentage of transactions is paid by UPI after cleaning? |
| VQ7.2 | Pie chart | Chart 10 (Section 5) | Compare | Is 'Unknown' bigger or smaller than 'Cash'? By how many transactions? |
| VQ7.3 | Pie chart | Chart 10 (Section 5) | Interpret | What does the 'Unknown' slice tell the regional head about data quality? |
| VQ7.4 | Pie chart | Chart 10 (Section 5) | Decide | Should the 'Unknown' slice be removed from the pie or kept? Give one reason. |
| VQ7.5 | Pie chart | Chart 10 (Section 5) | Fix / Choose | Redraw the same data as a bar chart. Which chart makes it easier to compare Wallet and Cash? |

### 📖 Definitions
| Term | Definition | Reliance example |
|---|---|---|
| **Visualisation** | A chart that shows data so patterns are easy to see. | Heatmap of blanks |
| **Read / Compare / Interpret / Decide / Fix** | Five levels of chart questions, from finding a number to improving the chart. | The VQ table |

## Practice Questions
1. Use the **Customer_Sample** sheet. Fill the blank `Rating` with the **mean** instead of the median. Does the average change? Which is safer, and why?
2. In `raw`, how many `Customer_Type` blanks are there? Which of the five methods would you use, and why?
3. Remove duplicates using only `Txn_ID` (`subset="Txn_ID"`). Do you get the same row count?
4. Which `Store_Format` has the most blank `Customer_Rating` values? Is that evidence of MCAR or MAR?
5. After cleaning, which month has the highest `Net_Amount`? Draw a line chart.
6. **Challenge:** Fill blank `Customer_Age` with the **mean age of each Payment_Mode** instead of the overall mean (hint: `groupby("Payment_Mode")["Customer_Age"].transform("mean")`). Why is this better for MAR data?